# LAB 3 #

JUAN PABLO PEREZ LOPEZ

In [1]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "Lab -3"

su=SparkUtils(spark_url,app_name)
su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 00:48:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
spakr

/opt/spark/work-dir


In [3]:
import os

print(os.listdir("/opt/spark/work-dir/data"))


['qcommece']


In [2]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating    
"""

column_types=[("Order_ID","long"),
              ("Company",'string'),
              ("City",'string'),
              ("Customer_Age",'int'),
              ("Order_Value",'float'),
              ("Delivery_Time_Min",'float'),
              ("Distance_Km",'float'),
              ("Items_Count",'float'),
              ("Product_Category",'string'),
              ("Payment_Method",'string'),
              ("Customer_Rating",'float'),
              ("Discount_Applied",'float'),
              ("Delivery_Partner_Rating",'float')   
]

qcommerce_schema = SparkUtils.generate_schema(column_types)

qcommerce_df = su._spark.read.option("header", "true").schema(qcommerce_schema).csv("/opt/spark/work-dir/data/qcommece")



qcommerce_df.printSchema()
print("Total records: ", qcommerce_df.count())
qcommerce_df.show(5)


root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)



Total records:  1000000
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.0|                    3.2|
|

In [4]:
qcommerce_null_count_df = SparkUtils.count_nulls(qcommerce_df)
qcommerce_null_count_df.show()

print("Total records with nulls: ", qcommerce_null_count_df.count())

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



Total records with nulls:  1


In [5]:
qcommerce_wo_nulls=qcommerce_df.na.drop()
print("Total records after dropping nulls: ", qcommerce_wo_nulls.count())

Total records after dropping nulls:  780992


In [21]:
qcommerce_without_null_fillna=qcommerce_df.fillna({
    'City': 'Unknown',
    'Items_Count': 0,
    'Customer_Rating': 0.0,
    'Delivery_Partner_Rating': 0.0
})
print("Total records after filling nulls with fillna: ", qcommerce_without_null_fillna.count())

Total records after filling nulls with fillna:  1000000


In [22]:
from pyspark.sql.functions import col,when,lit
qcommerce_without_null_fillna=qcommerce_without_null_fillna.withColumn('Code_Country',lit('IN'))
qcommerce_without_null_fillna.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [23]:
qcommerce_tax_df=qcommerce_without_null_fillna.withColumn('Paid_tax',col('Order_Value')*0.16)
qcommerce_tax_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|        Paid_tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10

### TASK 1 ###

In [27]:
#Create the new column with the conditions 


qcommerce_task1=qcommerce_tax_df.withColumn("Delivery_SLA",when(col('Delivery_Time_Min')<=15,'Fast').when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON_TIME").when(col('Delivery_Time_Min')>20,'Late'))
qcommerce_result1=qcommerce_task1
qcommerce_result1=qcommerce_result1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA").orderBy(col("Delivery_Time_Min").desc())
#qcommerce_task1.show()
qcommerce_result1.show()

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1807126|Jio Mart|Haridwar|             40.0|        Late|
| 1529779|Jio Mart|Haridwar|             40.0|        Late|
| 1165764|Jio Mart|Haridwar|           39.994|        Late|
| 1729503|Jio Mart|Haridwar|           39.994|        Late|
| 1610720|Jio Mart|Haridwar|           39.994|        Late|
| 1951122|Jio Mart|Haridwar|           39.988|        Late|
| 1975896|Jio Mart|Haridwar|           39.988|        Late|
| 1059671|Jio Mart|Haridwar|           39.982|        Late|
| 1594835|Jio Mart|Haridwar|           39.982|        Late|
| 1162804|Jio Mart|Haridwar|           39.982|        Late|
| 1826295|Jio Mart|Haridwar|           39.982|        Late|
| 1961544|Jio Mart|Haridwar|           39.976|        Late|
| 1360875|Jio Mart|Haridwar|           39.964|        Late|
| 1709566|Jio Mart|Haridwar|           3

### Task 2 ###

In [31]:

from pyspark.sql.functions import avg,count,col,when

qcommerce_task2=qcommerce_task1.withColumn('Effective_Order_Value',col('Order_Value') * (1 -col('Discount_Applied')))

qcommerce_task2=qcommerce_task2.withColumn("Price_Bucket",
                                       when(col('Effective_Order_Value')<200,'Low')
                                       .when((col('Effective_Order_Value')>=200) & (col("Effective_Order_Value") <=500),"MEDIUM")
                                       .when(col('Effective_Order_Value')>500,'HIGH')
                                       )
qcommerce_result2=qcommerce_task2

qcommerce_result2= qcommerce_result2.groupBy('Price_Bucket').agg(
    count('*').alias('Count_Price_Bucket'),
    avg("Effective_Order_Value").alias('Avg_Effective_Value')
).orderBy(col('Avg_Effective_Value').desc())


qcommerce_result2.show()


+------------+------------------+-------------------+
|Price_Bucket|Count_Price_Bucket|Avg_Effective_Value|
+------------+------------------+-------------------+
|        HIGH|            268953|  740.4337238893675|
|      MEDIUM|            210429|  358.0973233400432|
|         Low|            520618| 21.591204157544553|
+------------+------------------+-------------------+



### Task 3 ###

In [32]:
qcommerce_task3=qcommerce_task2.withColumn("Age_Group",
                                      when(col('Customer_Age')<25,'YOUNG')
                                      .when((col('Customer_Age')>=25) & (col('Customer_Age')<=44),'ADULT')
                                      .when(col('Customer_Age')>44,'SENIOR'))

qcommerce_task3_clean=qcommerce_task3.filter(
    (col('Customer_Age')>=0) &
    (col('Customer_Age')<=100)
)

qcommerce_result3=qcommerce_task3_clean.groupby('Age_Group','Product_Category').agg(
    count('*').alias('orders'),
    avg('Order_Value').alias('Average_Order_Value')
)

qcommerce_result3=qcommerce_result3.orderBy(
    col('Age_Group').asc(),
    col('Average_Order_Value').desc()
)

qcommerce_result3.show()



+---------+-------------------+------+-------------------+
|Age_Group|   Product_Category|orders|Average_Order_Value|
+---------+-------------------+------+-------------------+
|    ADULT|              Dairy| 68512|  573.4268899414931|
|    ADULT|          Household| 68110|  572.9259149188181|
|    ADULT|          Beverages| 67979|  572.0329877095578|
|    ADULT|          Groceries| 68291|  571.5250993844182|
|    ADULT|             Snacks| 68100|  570.3797974095505|
|    ADULT|Fruits & Vegetables| 67736|  569.8593599596651|
|    ADULT|      Personal Care| 68331|   569.512671998805|
|   SENIOR|              Dairy| 51025|   573.781117268945|
|   SENIOR|Fruits & Vegetables| 50642|  573.7244422534909|
|   SENIOR|             Snacks| 50924|   572.687851608881|
|   SENIOR|          Groceries| 51030|  572.2596391052539|
|   SENIOR|          Household| 50803|   571.172082385334|
|   SENIOR|      Personal Care| 50707|  571.1642560109325|
|   SENIOR|          Beverages| 50746|   568.14101323125

In [33]:
su._spark.stop()